# Week 5 — `groupby` and `merge`

`groupby` splits a table, computes something inside each group, and stacks the results. `merge` lines up two tables that share a key, and places their columns side by side.

The first groups are a four-row table you can check by eye. Then three Maddison files. The merge examples start with two tiny tables and end with a case where repeated keys multiply rows.

Run this notebook from `introduction-to-data-engineering`, from `week-05/materials`, or from `week-05/student_work`.


## A table small enough to check


In [ ]:
import pandas as pd

df_small = pd.DataFrame(
    {
        "country": ["Finland", "Finland", "Japan", "Japan"],
        "year": [2019, 2020, 2019, 2020],
        "gdp": [240, 250, 5100, 5000],
    }
)
df_small


## Basic `groupby`

`.size()` counts rows per country. `.min()` and `.max()` run inside each country. Finland's smallest GDP in this table is 240. Japan's is 5000.


In [ ]:
df_small.groupby("country").size()


In [ ]:
df_small.groupby("country")["gdp"].min()


In [ ]:
df_small.groupby("country")["gdp"].max()


`agg` names each result. `as_index=False` keeps `country` as a normal column, which a later merge can use as a key.


In [ ]:
summary_small = df_small.groupby("country", as_index=False).agg(
    first_year=("year", "min"),
    last_year=("year", "max"),
    min_gdp=("gdp", "min"),
    max_gdp=("gdp", "max"),
)
summary_small


## Three Maddison countries

Same `groupby`, now on Finland, Sweden, and Japan. Stack the files first, as in the concat notebook.


In [ ]:
from pathlib import Path


def find_existing(candidates):
    for path in candidates:
        if path.exists():
            return path
    searched = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Could not find the Maddison folder. Looked in:\n" + searched
    )


cwd = Path.cwd()
maddison_dir = find_existing(
    [
        cwd / "maddison_by_country",
        cwd / "materials/maddison_by_country",
        cwd / "week-05/materials/maddison_by_country",
        cwd / "../materials/maddison_by_country",
        cwd / "week-03/materials/maddison_by_country",
        cwd / "../../week-03/materials/maddison_by_country",
    ]
)
names = ["Finland", "Sweden", "Japan"]
frames = [pd.read_csv(maddison_dir / f"{name}.csv") for name in names]
df = pd.concat(frames, ignore_index=True)
df["Entity"].unique(), len(df)


In [ ]:
country_stats = df.groupby("Entity", as_index=False).agg(
    first_year=("Year", "min"),
    last_year=("Year", "max"),
    min_gdp=("GDP", "min"),
    max_gdp=("GDP", "max"),
)
country_stats


## In depth: the year of the min and the max

`min_gdp` is the value. `idxmin` finds the row, and the `Year` on that row is when it happened. Merge that year back onto `country_stats` on `Entity`.

Japan's latest year is 2022. Japan's largest GDP is in 2018. Finland and Sweden have their largest GDP in 2022, the last year in the file.


In [ ]:
low_labels = df.groupby("Entity")["GDP"].idxmin()
high_labels = df.groupby("Entity")["GDP"].idxmax()
year_of_min = df.loc[low_labels, ["Entity", "Year"]].rename(
    columns={"Year": "year_of_min_gdp"}
)
year_of_max = df.loc[high_labels, ["Entity", "Year"]].rename(
    columns={"Year": "year_of_max_gdp"}
)
pd.merge(country_stats, year_of_min, on="Entity")


In [ ]:
stats = pd.merge(country_stats, year_of_min, on="Entity")
stats = pd.merge(stats, year_of_max, on="Entity")
stats


## Basic `merge`

`concat` stacks rows. `merge` matches a key and adds columns from the other table.

Both tables below use the same countries, so every row finds a match.


In [ ]:
population = pd.DataFrame(
    {
        "country": ["Finland", "Japan", "Bhutan"],
        "population": [5_620_000, 123_975_000, 790_000],
    }
)
gdp = pd.DataFrame(
    {
        "country": ["Finland", "Japan", "Bhutan"],
        "gdp": [300, 4200, 3.0],
    }
)
pd.merge(population, gdp, on="country")


## `inner`, `left`, and `outer`

Tuvalu is only in `population`. Suriname is only in `gdp_gap`.

| `how` | rows you keep |
| --- | --- |
| `inner` | keys that appear on both sides |
| `left` | every row of the left table |
| `outer` | every key from either table |

A key that is missing on one side shows up as `NaN` in that side's columns. `how="right"` keeps every row of the right table.


In [ ]:
population_gap = pd.DataFrame(
    {
        "country": ["Finland", "Japan", "Tuvalu"],
        "population": [5_620_000, 123_975_000, 10_000],
    }
)
gdp_gap = pd.DataFrame(
    {
        "country": ["Finland", "Japan", "Suriname"],
        "gdp": [300, 4200, 4.0],
    }
)
pd.merge(population_gap, gdp_gap, on="country", how="inner")


In [ ]:
pd.merge(population_gap, gdp_gap, on="country", how="left")


In [ ]:
pd.merge(population_gap, gdp_gap, on="country", how="outer")


## In depth: repeated keys multiply rows

The left table has two `Blue` rows. The right table has two `Blue` rows. Each left Blue matches each right Blue, so 2 × 2 = 4 Blue rows, plus one Red row.


In [ ]:
students = pd.DataFrame(
    {"color": ["Blue", "Blue", "Red"], "student": ["A", "B", "C"]}
)
shades = pd.DataFrame(
    {"color": ["Blue", "Blue", "Red"], "shade": ["sky", "sea", "stop"]}
)
color_merge = pd.merge(students, shades, on="color")
color_merge


In [ ]:
len(students), len(shades), len(color_merge)


## Practice

`country_stats` is already in memory (Finland, Sweden, Japan).

1. Count the rows per `Entity` in `df` with `groupby` and `.size()`.
2. Left-merge the capital table below onto `country_stats`. Which entity has an empty `min_gdp`?
3. Repeat that merge with `how="inner"`. Which entity is dropped?

```python
capitals = pd.DataFrame(
    {
        "Entity": ["Finland", "Sweden", "Japan", "Atlantis"],
        "capital": ["Helsinki", "Stockholm", "Tokyo", "Unknown"],
    }
)
```


In [ ]:
# Practice 1


In [ ]:
# Practice 2


In [ ]:
# Practice 3


## Quiz

**1.** What does `df_small.groupby("country")["gdp"].min()` return?

- A) One smallest GDP for the whole table
- B) The smallest GDP inside each country
- C) The row where GDP is smallest
- D) The number of countries

**2.** `pd.merge(population, gdp, on="country")` keeps which rows when some countries appear on only one side?

- A) Every row of `population`
- B) Countries that appear in both tables
- C) Every country from either table
- D) Only the first row of each table

**3.** You want every row of the left table, with `NaN` where the right table has no match. Which `how` is that?

- A) `"inner"`
- B) `"left"`
- C) `"outer"`
- D) `"cross"`

**4.** `students` has two rows with color Blue. `shades` has two rows with color Blue. An inner merge on `color` produces how many Blue rows?

- A) 2
- B) 3
- C) 4
- D) 8


## Answer key

For the instructor. Remove this section before sharing the notebook if the quiz should be unseen.

**Quiz:** 1 B, 2 B, 3 B, 4 C.

**Practice:**

1. Finland 167, Sweden 206, Japan 146.
2. Atlantis. It is only in `capitals`, so a left merge keeps it and `min_gdp` is `NaN`. Put `capitals` on the left.
3. Atlantis is dropped. Inner keeps Finland, Sweden, and Japan.

```python
df.groupby("Entity").size()

pd.merge(capitals, country_stats, on="Entity", how="left")
pd.merge(capitals, country_stats, on="Entity", how="inner")
```
